# Install librarys
Install packages need to train mario agent.

In [ ]:
import sys
!{sys.executable} -m pip install gymnasium==0.29.1
!{sys.executable} -m pip install gym-super-mario-bros==7.4.0
!{sys.executable} -m pip install gym==0.25.2
!{sys.executable} -m pip install imageio-ffmpeg
!{sys.executable} -m pip install imageio
# !pip install torchvision
!{sys.executable} -m pip install opencv-python-headless
!{sys.executable} -m pip install numpy==1.26.4

# import packages

In [ ]:
from PIL import Image
from collections import deque
from datetime import datetime
from pathlib import Path
import copy
import cv2
import imageio
import numpy as np
import random, os
import torch
from torch import nn
import torch.nn.functional as F
import torch.multiprocessing as mp
#import multiprocessing as mp
from torchvision import transforms as T
import gc

# Gym is an OpenAI toolkit for RL
import gym
from gym.spaces import Box
from gym.wrappers import FrameStack

# NES Emulator for OpenAI Gym
from nes_py.wrappers import JoypadSpace

# Super Mario environment for OpenAI Gym
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, COMPLEX_MOVEMENT, RIGHT_ONLY

# Create hyperparammeters
Config hyperparammeters, just change it

In [ ]:
#class DictWrapper create by Chatgpt
class DictWrapper:
    def __init__(self, dictionary):
        self._dict = dictionary

    def __getattr__(self, item):
        if item in self._dict:
            return self._dict[item]
        else:
            raise AttributeError(f"'DictWrapper' object has no attribute '{item}'")

config = {
    'num_envs': 16, #if available, try 256 as r2d2 paper
    'save_model_step': int(1e5),
    'save_figure_step': 400,
    'learn_step': 4, #paper use 52
    'total_step_or_episode': 'step',
    'total_step': int(1e7),
    'total_episode': None,
    'batch_size': 16, #paper use 64
    'save_dir': "",
    'gamma': 0.997,
    'learning_rate': 1e-4,
    'state_dim': (1, 84, 84),
    'action_dim': 12,#12 for complex, 7 for simple
    'max_grad_norm': 40,
    'world': 1,
    'stage': 1,
    'action_type': 'complex',
    'loss_type': "mse", #"huber"
    'target_update_freq': 2500,
    'replay_buffer_size': int(1e5),
    'replay_buffer_sample_size': int(4e6)+85,
    'per_eps': 1e-2,
    'per_alpha': 0.9,
    'per_beta': 0.6,
    'eta': 0.9,
    'use_layer_init': True,
    'additional_bonus_state_8_4_option': "no", #"right_pipe",
    'm': 40, 
    'l': 40, 
    'n': 5,
    'start_learning_step': 50000, 
    'start_learning_sequence': 6250,
}

config = DictWrapper(config)

In [ ]:
gym.__version__

# Define environment
## Create a custom environment, We need:
- SkipFrame: Because the episode is very long, we only need to repeat actions sometimes in this environment. We repeat each action 4 times and skip the first 3 frames (returning the 4th frame). We also sum the rewards from all 4 frames.
- GrayScaleResizeObservation: Convert the state to grayscale (from RGB to a gray image) and resize it to 84x84 pixels.
- NoopResetEnv: When resetting the environment, we perform random actions before starting the environment. This is similar to the Atari strategy. When resetting, we randomly choose num_noops actions between 0 and noop_max and perform num_noops random actions. If the random actions lead to a terminal state, we reset and continue performing random actions. I set noop_max to 30, similar to Atari.
- CustomRewardAndDoneEnv
    - I noticed that many people train Mario using this custom reward system, so I copied it. The system adds 50 reward points if the agent solves the stage and subtracts 50 reward points if the agent dies. The reward is divided by 10. I set done to True if Mario dies, instead of the default setting where Mario loses all lives.
    - Stage 4-2: Subtract 50 reward points if Mario moves on top of the map (y_pos >= 255).
    - Stages 4-4 and 7-4: Set done = True when Mario goes the wrong way and subtract 50 reward points as a penalty. If Mario takes the correct path but the map still loops (a known bug), I set done = True but do not apply a penalty.
    - Stage 8-4: Set done = True when Mario goes the wrong way and subtract 50 reward points as a penalty (similar to stages 4-4 and 7-4). This map has a particularly difficult section where Mario needs to find a hidden brick.
    - Stage 8-4: if config.additional_bonus_state_8_4_option = 'right_pipe', Add 50 reward points when Mario successfully takes the correct path.
    - Stages 4-4 and 8-4: give a -0.1 reward every step.


## About reward system:
- Set done to True when Mario dies: This is the most important aspect because, in the default reward system, Mario still gains a reward by just moving right. If Mario dies, the agent doesn't lose total rewards and can continue moving right (in the new life) to get more rewards. This is the easiest way for the agent to earn rewards, and it can learn to exploit this trick.
- Penalty of -50 reward when Mario dies: This is necessary to speed up Mario's training. Without this penalty, the agent may struggle to complete more difficult stages.
- Reward of 50 when reaching the flag: This encourages Mario to train faster and overcome difficult sections in harder stages.
- Changing the penalty and flag reward to more or less than 50 doesn't make a significant difference, so I haven't changed it.
- Divide rewards by 10: I believe this reduces the total rewards and helps the agent learn a better strategy, but I'm not entirely sure how necessary this is. I simply followed an existing approach.
- With Stage 4-2: I noticed that the agent can earn more rewards when Mario goes to the warp zone, but Mario can't win this stage using the warp zone because the reward system gives negative rewards when Mario moves left. Therefore, I added a penalty when Mario moves to the top of the map.
- Use FrameStack to stack the latest 4 frames as observation.
- With Stage 4-4 and 7-4:
    - Since this map has a wrong path, Mario can enter a loop where the reward increases indefinitely. To prevent this, I set done = True and assign a negative penalty reward.
    - I also give a negative reward to prevent Mario from taking the wrong path.
    - Another strategy is to give a negative reward without setting done = True (as in 4-2). However, this strategy doesn't work due to a bug in this map.
    - Even when Mario is on the correct path, sometimes he still enters the loop. To handle this, I set done = True every time Mario enters the loop (checked by x_pos and max_x_pos).
- Stage 4-4: Assign a -0.1 reward for every step: This prevents Mario from getting stuck. This map has a section where Mario needs to move left, but moving left incurs a negative reward in the default system. If Mario moves right, he takes the wrong path, causing the episode to end with a negative reward. To keep Mario moving, I added a negative reward for every step.
- Stage 8-4:
    - Assign a -0.1 reward for every step: This prevents Mario from getting stuck. This encourages Mario to move when stuck at a particular section (info["x_pos"] > 2440 and info["x_pos"] <= 2500). Since moving right can lead Mario down the wrong path, the agent often learns to do nothing to avoid losing rewards. This penalty encourages exploration to find the hidden brick.
    - Same as in 4-4 and 7-4, I set done = True and assign a -50 penalty when the agent moves in the wrong direction.
    - If config.additional_bonus_state_8_4_option = 'right_pipe', Add a 50 reward when the agent goes the correct way (avoiding the wrong path).
    - Since this map has many locations with overlapping x_pos values (especially in the underwater section where x_pos is reset to 1), be careful when modifying the custom reward system.

In [ ]:
# Initialize Super Mario environment (in v0.26 change render mode to 'human' to see results on the screen)
if gym.__version__ < '0.26':
    env = gym_super_mario_bros.make(f"SuperMarioBros-{config.world}-{config.stage}-v0", new_step_api=True)
else:
    env = gym_super_mario_bros.make(f"SuperMarioBros-{config.world}-{config.stage}-v0", render_mode='rgb', apply_api_compatibility=True)

env = JoypadSpace(env, COMPLEX_MOVEMENT)
print(env.action_space)

env.reset()
next_state, reward, done, trunc, info = env.step(action=0)
print(f"{next_state.shape},\n {reward},\n {done},\n {info}")

class SkipFrame(gym.Wrapper):
    def __init__(self, env, skip):
        """Return only every `skip`-th frame"""
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        """Repeat action, and sum reward"""
        total_reward = 0.0
        for i in range(self._skip):
            # Accumulate reward and repeat the same action
            obs, reward, done, trunk, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, trunk, info

class GrayScaleResizeObservation(gym.ObservationWrapper):
    def __init__(self, env, shape):
        super().__init__(env)
        if isinstance(shape, int):
            self.shape = (shape, shape)
        else:
            self.shape = tuple(shape)

        obs_shape = self.shape + self.observation_space.shape[2:]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)
        self.current_state = None

    def observation(self, observation):
        self.current_state = np.array(observation)
        observation = cv2.cvtColor(observation, cv2.COLOR_RGB2GRAY)
        observation = cv2.resize(observation, self.shape, interpolation=cv2.INTER_AREA)
        observation = observation.astype(np.uint8).reshape(-1, observation.shape[0], observation.shape[1])
        return observation

class NoopResetEnv(gym.Wrapper):
    def __init__(self, env, noop_max=30):
        super(NoopResetEnv, self).__init__(env)
        self.noop_max = noop_max

    def reset(self, **kwargs):
        """Do no-op action for a number of steps in [1, noop_max]."""
        obs = self.env.reset(**kwargs)
        noops = np.random.randint(0, self.noop_max, (1, ))[0]
        for _ in range(noops):
            action = self.env.action_space.sample()
            obs, _, done, _, _ = self.env.step(action)
            if done:
                obs = self.env.reset(**kwargs)
        return obs

    def step(self, ac):
        obs, reward, done, trunk, info = self.env.step(ac)
        return obs, reward, done, trunk, info

class CustomRewardAndDoneEnv(gym.Wrapper):
    def __init__(self, env=None, world=1, stage=1, additional_bonus_state_8_4_option = "no"):
        super(CustomRewardAndDoneEnv, self).__init__(env)
        self.current_score = 0
        self.current_x = 0
        self.old_x = -1
        self.current_x_count = 0
        self.max_x = 0
        self.world = world
        self.stage = stage
        if self.world == 8 and self.stage == 4:
            self.sea_map = False
            self.additional_bonus_state_8_4_option = additional_bonus_state_8_4_option

    def reset(self, **kwargs):
        self.current_score = 0
        self.current_x = 0
        self.old_x = -1
        self.current_x_count = 0
        self.max_x = 0
        if self.world == 8 and self.stage == 4:
            self.sea_map = False
        return self.env.reset(**kwargs)

    def step(self, action):
        state, reward, done, trunc, info = self.env.step(action)

        if info['world'] != self.world or info['stage'] != self.stage:
            done = True
            info["flag_get"] = True

        if (info['x_pos'] - self.current_x) == 0:
            self.current_x_count += 1
        else:
            self.current_x_count = 0
        if info["flag_get"]:
            reward += 50
            done = True
        if done and info["flag_get"] == False and info["time"] != 0:
            reward -= 50
            done = True
        self.current_x = info["x_pos"]

        if self.world == 7 and self.stage == 4:
            if (506 <= info["x_pos"] <= 832 and info["y_pos"] > 127) or (
                    832 < info["x_pos"] <= 1064 and info["y_pos"] < 80) or (
                    1113 < info["x_pos"] <= 1464 and info["y_pos"] < 191) or (
                    1579 < info["x_pos"] <= 1943 and info["y_pos"] < 191) or (
                    1946 < info["x_pos"] <= 1964 and info["y_pos"] >= 191) or (
                    1984 < info["x_pos"] <= 2060 and (info["y_pos"] >= 191 or info["y_pos"] < 127)) or (
                    2114 < info["x_pos"] < 2440 and info["y_pos"] < 191):
                reward -= 50
                done = True
            if done == False and info["x_pos"] < self.max_x - 100:
                done = True
        if self.world == 4 and self.stage == 4:
            if (info["x_pos"] <= 1500 and info["y_pos"] < 127) or (
                    1588 <= info["x_pos"] < 2380 and info["y_pos"] >= 127):
                reward -= 50
                done = True
            if done == False and info["x_pos"] < self.max_x - 100:
                done = True
            if done == False:
                reward -= 0.1
        if self.world == 4 and self.stage == 2 and done == False and info['y_pos'] >= 255:
            reward -= 50
        if self.world == 8 and self.stage == 4:
            if info["x_pos"] > 2440 and info["x_pos"] <= 2500:
                done = True
                reward -= 50 #100 #50
            if info["x_pos"] >= 3675 and info["x_pos"] <= 3700:
                done = True
                reward -= 50

            if info["x_pos"] < self.max_x - 200:
                if self.max_x >= 1240 and self.max_x <= 1310: #solved bug because x_pos duplicated
                    if info["x_pos"] >= 320:
                        done = True
                        reward -= 50

            if info["x_pos"] < self.old_x - 200:
                if info["x_pos"] >= 312-5 and info["x_pos"] <= 312+5:
                    done = True
                    reward -= 50
                elif info["x_pos"] >= 56-5 and info["x_pos"] <= 56+5 and self.max_x > 3645 and self.sea_map == False:
                    if self.additional_bonus_state_8_4_option == 'right_pipe':
                        reward += 50
                    self.sea_map = True
            if self.additional_bonus_state_8_4_option == 'right_pipe':
                if info["x_pos"] > self.max_x + 100:
                    reward += 50
            if done == False:
                reward -= 0.1
        self.max_x = max(self.max_x, self.current_x)
        self.current_score = info["score"]
        self.old_x = self.current_x

        return state, reward / 10., done, trunc, info

# Create MultipleEnvironments
MultipleEnvironments use multi-processing to parallel running.

Because in the training process, we need to reset the environment when the agent reaches the terminal state. But if we will do it in parallel, then I don't want to check each environment and reset (by loop) or create a new function that parallels check and reset all environments. Then I reset the environment if done = True in step function and set next_state = env.reset(). Then in training, we just set state = next_state (next_state is reset state if done = True)

In [ ]:
#modify from https://github.com/uvipen/Super-mario-bros-PPO-pytorch/blob/master/src/env.py
def create_env(world, stage, action_type, additional_bonus_state_8_4_option, test=False):
    if gym.__version__ < '0.26':
        env = gym_super_mario_bros.make(f"SuperMarioBros-{world}-{stage}-v0", new_step_api=True)
    else:
        env = gym_super_mario_bros.make(f"SuperMarioBros-{world}-{stage}-v0", render_mode='rgb', apply_api_compatibility=True)

    if action_type == "right":
        action_type = RIGHT_ONLY
    elif action_type == "simple":
        action_type = SIMPLE_MOVEMENT
    else:
        action_type = COMPLEX_MOVEMENT

    env = JoypadSpace(env, action_type)

    if test == False:
        env = NoopResetEnv(env)
    env = SkipFrame(env, skip=4)
    env = CustomRewardAndDoneEnv(env, world, stage, additional_bonus_state_8_4_option)
    env = GrayScaleResizeObservation(env, shape=84)
    # if gym.__version__ < '0.26':
    #     env = FrameStack(env, num_stack=4, new_step_api=True)
    # else:
    #     env = FrameStack(env, num_stack=4)
    return env

class MultipleEnvironments:
    def __init__(self, world, stage, action_type, num_envs, additional_bonus_state_8_4_option):
        self.agent_conns, self.env_conns = zip(*[mp.Pipe(duplex=True) for _ in range(num_envs)])
        self.envs = [create_env(world, stage, action_type, additional_bonus_state_8_4_option) for _ in range(num_envs)]

        for index in range(num_envs):
            process = mp.Process(target=self.run, args=(index,))
            process.start()
            self.env_conns[index].close()

    def run(self, index):
        self.agent_conns[index].close()
        while True:
            request, action = self.env_conns[index].recv()
            if request == "step":
                next_state1, reward, done, trunc, info = self.envs[index].step(action)
                if done:
                    next_state2 = self.envs[index].reset()
                else:
                    next_state2 = next_state1
                self.env_conns[index].send((next_state1, next_state2, reward, done, trunc, info))
            elif request == "reset":
                self.env_conns[index].send(self.envs[index].reset())
            else:
                raise NotImplementedError

    def step(self, actions):
        [agent_conn.send(("step", act)) for agent_conn, act in zip(self.agent_conns, actions)]
        next_state1, next_state2, rewards, dones, truncs, infos = zip(*[agent_conn.recv() for agent_conn in self.agent_conns])
        return next_state1, next_state2, rewards, dones, truncs, infos

    def reset(self):
        [agent_conn.send(("reset", None)) for agent_conn in self.agent_conns]
        states = [agent_conn.recv() for agent_conn in self.agent_conns]
        return states

In [ ]:
mp.cpu_count()

# Create Episode_memory
Save data in 1 episode. I just push data to per after end of episode then I need save local episode data before push it to per.

In [ ]:
class Episode_memory():
    def __init__(self, num_envs, m, l, n):
        self.num_envs = num_envs
        self.m = m
        self.l = l
        self.n = n
        self.lens = [self.m for _ in range(self.num_envs)]
        self.reset()

    def save(self, states, actions, rewards, dones, values, target_values, h, c, last_actions, last_rewards):
        for i in range(self.num_envs):
            self.lens[i] += 1
            self.states[i].append(states[i])
            self.actions[i].append(actions[i])
            self.rewards[i].append(rewards[i])
            self.dones[i].append(dones[i])
            self.values[i].append(values[i].detach().cpu())
            self.target_values[i].append(target_values[i].detach().cpu())
            self.h[i].append(h[i].detach().cpu())
            self.c[i].append(c[i].detach().cpu())
            self.last_actions[i].append(last_actions[i].detach().cpu())
            self.last_rewards[i].append(last_rewards[i].detach().cpu())

    def reset(self, env_i = None):
        if env_i is None:
            self.lens = [self.m for _ in range(self.num_envs)]
            self.states = [[] for _ in range(self.num_envs)]
            self.actions = [[] for _ in range(self.num_envs)]
            self.rewards = [[] for _ in range(self.num_envs)]
            self.dones = [[] for _ in range(self.num_envs)]
            self.values = [[] for _ in range(self.num_envs)]
            self.target_values = [[] for _ in range(self.num_envs)]
            self.h = [[] for _ in range(self.num_envs)]
            self.c = [[] for _ in range(self.num_envs)]
            self.last_actions = [[] for _ in range(self.num_envs)]
            self.last_rewards = [[] for _ in range(self.num_envs)]
        else:
            self.lens[env_i] = self.m
            self.states[env_i] = []
            self.actions[env_i] = []
            self.rewards[env_i] = []
            self.dones[env_i] = []
            self.values[env_i] = []
            self.target_values[env_i] = []
            self.h[env_i] = []
            self.c[env_i] = []
            self.last_actions[env_i] = []
            self.last_rewards[env_i] = []

    def get_data(self, env_i = None):
        if env_i is None:
            return self.states, self.actions, self.rewards, self.dones, self.values, self.target_values, self.h, self.c, self.last_actions, self.last_rewards
        return (self.states[env_i], self.actions[env_i], self.rewards[env_i],
                self.dones[env_i], self.values[env_i], self.target_values[env_i], self.h[env_i], self.c[env_i],
                self.last_actions[env_i], self.last_rewards[env_i])

# SumTree
I copy from [Howuhh prioritized_experience_replay](https://github.com/Howuhh/prioritized_experience_replay/blob/main/memory/buffer.py)

In [ ]:
# The ‘sum-tree’ data structure used here is very similar in spirit to the array representation
# of a binary heap. However, instead of the usual heap property, the value of a parent node is
# the sum of its children. Leaf nodes store the transition priorities and the internal nodes are
# intermediate sums, with the parent node containing the sum over all priorities, p_total. This
# provides a efficient way of calculating the cumulative sum of priorities, allowing O(log N) updates
# and sampling. (Appendix B.2.1, Proportional prioritization)

# Additional useful links
# Good tutorial about SumTree data structure:  https://adventuresinmachinelearning.com/sumtree-introduction-python/
# How to represent full binary tree as array: https://stackoverflow.com/questions/8256222/binary-tree-represented-using-array
class SumTree:
    def __init__(self, size):
        self.nodes = [0] * (2 * size - 1)
        self.data = [None] * size

        self.size = size
        self.count = 0
        self.real_size = 0

    @property
    def total(self):
        return self.nodes[0]

    def update(self, data_idx, value):
        idx = data_idx + self.size - 1  # child index in tree array
        change = value - self.nodes[idx]

        self.nodes[idx] = value

        parent = (idx - 1) // 2
        while parent >= 0:
            self.nodes[parent] += change
            parent = (parent - 1) // 2

    def add(self, value, data):
        self.data[self.count] = data
        self.update(self.count, value)

        self.count = (self.count + 1) % self.size
        self.real_size = min(self.size, self.real_size + 1)

    def get(self, cumsum):
        assert cumsum <= self.total

        idx = 0
        while 2 * idx + 1 < len(self.nodes):
            left, right = 2*idx + 1, 2*idx + 2

            if cumsum <= self.nodes[left]:
                idx = left
            else:
                idx = right
                cumsum = cumsum - self.nodes[left]

        data_idx = idx - self.size + 1

        return data_idx, self.nodes[idx], self.data[data_idx]

    def __repr__(self):
        return f"SumTree(nodes={self.nodes.__repr__()}, data={self.data.__repr__()})"

# PER
Edit from [Howuhh PER](https://github.com/Howuhh/prioritized_experience_replay/blob/main/memory/buffer.py). Use Gemini 3.5 Flash and Gemini 3.1 Pro to improve memory usage for episode saving.

In [ ]:
class R2D2ReplayBuffer:
    def __init__(self, state_dim, action_dim, buffer_size=100000, sample_size=4000000, l=40, m=40, n=5, eps=1e-2, alpha=0.6, beta=0.4):
        self.tree = SumTree(size=buffer_size)

        self.action_dim = action_dim

        # PER Hyperparameters
        self.eps = eps
        self.alpha = alpha
        self.beta = beta
        self.max_priority = eps

        self.l = l  # Sequence length for training (40 steps)
        self.m = m  # Burn-in length for init lstm (40 steps)
        self.n = n  # N-step lookahead to calculate Target Q
        self.seq_len = m + l + n

        # per size
        self.sample_size = int(sample_size)  # 4e6 observations
        self.size = int(buffer_size)        # 1e5 sequences

        self.states = np.zeros((self.sample_size, *state_dim), dtype=np.uint8)
        self.actions = np.zeros((self.sample_size,), dtype=np.int64)
        self.rewards = np.zeros((self.sample_size,), dtype=np.float32)
        self.dones = np.zeros((self.sample_size,), dtype=np.float32)

        self.last_actions = np.zeros((self.sample_size, self.action_dim), dtype=np.float32)
        self.last_rewards = np.zeros((self.sample_size,), dtype=np.float32)

        self.hs = np.zeros((self.size, 512), dtype=np.float32)
        self.cs = np.zeros((self.size, 512), dtype=np.float32)

        self.burnin_idx = np.zeros((self.size,), dtype=np.int64)
        self.start_idx = np.zeros((self.size,), dtype=np.int64)
        self.end_idx = np.zeros((self.size,), dtype=np.int64)

        self.global_sample_count = 0  
        self.count = 0                
        self.real_size = 0            

        self.active = np.zeros((self.size,), dtype=np.bool_)
        self.active_size = 0

        self.allocate_mem()

    def allocate_mem(self):
        self.states += 0
        self.actions += 0
        self.rewards += 0
        self.dones += 0
        self.hs += 0
        self.cs += 0
        self.last_actions += 0
        self.last_rewards += 0

    def invalidate_overwritten_sequences(self):
        min_valid_global_idx = max(0, self.global_sample_count - self.sample_size)

        invalid = self.active & (self.burnin_idx < min_valid_global_idx)
        invalid_idxs = np.flatnonzero(invalid)

        for idx in invalid_idxs:
            self.tree.update(int(idx), 0.0)

        self.active[invalid_idxs] = False
        self.active_size -= len(invalid_idxs)

    def add(self, transition, burnin_idx_arr, start_idx_arr, end_idx_arr, priorities):
        state, action, reward, done, h, c, last_actions, last_rewards = transition
        episode_len = len(state)

        flat_start = self.global_sample_count % self.sample_size
        flat_end = flat_start + episode_len

        if flat_end <= self.sample_size:
            self.states[flat_start:flat_end] = state
            self.actions[flat_start:flat_end] = action
            self.rewards[flat_start:flat_end] = reward
            self.dones[flat_start:flat_end] = done
            self.last_actions[flat_start:flat_end] = last_actions
            self.last_rewards[flat_start:flat_end] = last_rewards
        else:
            first_part_len = self.sample_size - flat_start

            self.states[flat_start:] = state[:first_part_len]
            self.actions[flat_start:] = action[:first_part_len]
            self.rewards[flat_start:] = reward[:first_part_len]
            self.dones[flat_start:] = done[:first_part_len]
            self.last_actions[flat_start:] = last_actions[:first_part_len]
            self.last_rewards[flat_start:] = last_rewards[:first_part_len]

            second_part_len = episode_len - first_part_len
            self.states[:second_part_len] = state[first_part_len:]
            self.actions[:second_part_len] = action[first_part_len:]
            self.rewards[:second_part_len] = reward[first_part_len:]
            self.dones[:second_part_len] = done[first_part_len:]
            self.last_actions[:second_part_len] = last_actions[first_part_len:]
            self.last_rewards[:second_part_len] = last_rewards[first_part_len:]

        for i in range(len(start_idx_arr)):

            if self.active[self.count]:
                self.active[self.count] = False
                self.active_size -= 1

            self.tree.add(self.max_priority, self.count)
            self.update_priorities([self.count], [priorities[i]])

            self.active[self.count] = True
            self.active_size += 1

            self.burnin_idx[self.count] = self.global_sample_count + int(burnin_idx_arr[i])
            self.start_idx[self.count] = self.global_sample_count + int(start_idx_arr[i])
            self.end_idx[self.count] = self.global_sample_count + int(end_idx_arr[i])

            self.hs[self.count] = h[i]
            self.cs[self.count] = c[i]

            self.count = (self.count + 1) % self.size
            self.real_size = min(self.size, self.real_size + 1)

        self.global_sample_count += episode_len
        self.invalidate_overwritten_sequences()

    def sample(self, batch_size):
        assert self.active_size >= batch_size, "Buffer chưa tích lũy đủ số lượng chuỗi tối thiểu."
        assert self.tree.total > 0

        sample_idxs, tree_idxs = [], []
        priorities = np.zeros((batch_size,))

        segment = self.tree.total / batch_size
        for i in range(batch_size):
            while True:
                a, b = segment * i, segment * (i + 1)
                cumsum = random.uniform(a, b)
                tree_idx, priority, sample_idx = self.tree.get(cumsum)

                if not self.active[sample_idx]:
                    continue

                priorities[i] = priority
                tree_idxs.append(tree_idx)
                sample_idxs.append(sample_idx)
                break

        probs = priorities / (self.tree.total + 1e-8)
        weights = (self.active_size * probs) ** -self.beta
        weights = weights / (weights.max() + 1e-8)

        base_idx_raw = self.start_idx[sample_idxs] - self.m
        base_idx = base_idx_raw % self.sample_size
        gather_idx = (base_idx[:, None] + np.arange(self.seq_len)[None, :]) % self.sample_size

        states  = self.states[gather_idx]
        actions = self.actions[gather_idx]
        rewards = self.rewards[gather_idx]
        dones   = self.dones[gather_idx]
        hs      = self.hs[sample_idxs]
        cs      = self.cs[sample_idxs]
        last_actions = self.last_actions[gather_idx]
        last_rewards = self.last_rewards[gather_idx]

        masks = np.ones((batch_size, self.seq_len), dtype=np.float32)
        pos = np.arange(self.seq_len)

        valid_burnin = self.start_idx[sample_idxs] - self.burnin_idx[sample_idxs]
        missing_burnin = np.maximum(0, self.m - valid_burnin)
        masks[pos[None, :] < missing_burnin[:, None]] = 0

        valid_tail = self.end_idx[sample_idxs] - self.start_idx[sample_idxs] + 1
        missing_tail = np.maximum(0, (self.l + self.n) - valid_tail)
        masks[pos[None, :] >= (self.seq_len - missing_tail)[:, None]] = 0

        batch = (
            np.swapaxes(states, 0, 1),
            np.swapaxes(actions, 0, 1),
            np.swapaxes(rewards, 0, 1),
            np.swapaxes(dones, 0, 1),
            hs, cs,
            np.swapaxes(last_actions, 0, 1),
            np.swapaxes(last_rewards, 0, 1),
            np.swapaxes(masks, 0, 1)
        )
        return batch, weights, tree_idxs

    def update_priorities(self, data_idxs, priorities):
        for data_idx, priority in zip(data_idxs, priorities):
            priority = (float(priority) + self.eps) ** self.alpha
            self.tree.update(data_idx, priority)
            self.max_priority = max(self.max_priority, priority)

# Create agent
The agent includes 10 main functions:

## train
The train function trains the agent through many episodes:
* Reset the initial state.
* Initialize (h, c), (target_h, target_c) as zeros.
* Create last_action and last_reward (set both to 0).
* Create Episode_memory
* Copy target_model from model (call update_target_model function)
* Loop until the agent wins this stage or reaches the maximum episode/step:
  * predict action, Q value, target_q, new_h, new_c, new_target_h, new_target_c for the current state (call select_action function)
  * log all information to memory
  * (h, c) = (new_h, new_c), (target_h, target_c) = (new_target_h, new_target_c)
  * reset h, c, target_h, target_c to zero if a terminal state is reached
  * calculate one-hot last_action (set to zero if $state_0$) and last_reward
  * if any env end it'episode, call save_episode function to push this episode data to per
  * train the agent every learn_step (learn function)
  * evaluate the agent every save_figure_step (save_figure function)
  * set state = next_state_ (if done, next_state_ is $state_0$ of the next episode; next_state is the terminal state; otherwise, next_state_ = next_state).

## select_action
This function samples an action, calculate Q value, new_h, new_c, target_q, new_target_h, new_target_c:
* Calculate Q value, new_h, new_c by online model.
* Calculate target_q, new_target_h, new_target_c by target model.
* Use epsilon greedy to sample action from Q value and epsilon.

## update_target_model
This function copy weights from online model to target model

## value_rescale
This function transform value (to scale/transform value). R2D2 use this to tranform value instead of normalize/clip/scale reward/return/value as old paper (DQN use sign clip, PPO often normalize reward by rms std of reward/return, ...).

## inverse_value_rescale
This function transform input back to true value (inverse of value_rescale function).

## calculate_target
This function calculate n_step returns from episode data.

## calculate_priorities
This function calculate per priority for 1 episode.

## save_episode
This function push episode data to per:
* Use calculate_target function to calculate target for each step (each state) in this episode.
* Use calculate_priorities function to calculate priority of this episode.
* Push this episode to per (call per.add)

## save_figure
This function evaluates the agent and saves the agent/video if the agent achieves a better total reward:
* reset the environment.
* initialize (h, c) as zeros and reset the environment.
* create last_action and last_reward (set both to 0).
* loop until the agent reaches a terminal state.
  * predict Q value, h, c from the model
  * get action = argmax(Q value)
  * execute the action in the environment to obtain next_state, reward, info, and done
  * calculate one-hot last_action and last_reward
  * if total_reward > best test total reward or the agent completes this stage, save the model and video
  * if the agent completes this stage, stop training.

## learn
This function trains the agent using experiences stored in memory:
* get all information from memory
* calculate Q value and target_q (stop gradient for first burn-in steps and last n_step steps, don't use there steps to train model)
* calculate target
* calculate loss and backward
* normalize gradients
* update model
* update priorities for there sequences.

In [ ]:
class Agent():
    def __init__(self, world, stage, action_type, envs, num_envs, additional_bonus_state_8_4_option,
                 state_dim, action_dim, save_dir, save_model_step,
                 save_figure_step, learn_step, total_step_or_episode, total_step, total_episode, model, target_model,
                 gamma, learning_rate, max_grad_norm, target_update_freq, replay_buffer_size, per,
                 per_eps, per_alpha, per_beta, eta, m, l, n, batch_size, loss_type, epsilons, start_learning_step,
                 start_learning_sequence, device):
        self.world = world
        self.stage = stage
        self.action_type = action_type

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.save_dir = save_dir
        self.learn_step = learn_step
        self.total_step_or_episode = total_step_or_episode
        self.total_step = total_step
        self.total_episode = total_episode
        self.target_update_freq = target_update_freq

        self.current_step = 0
        self.current_episode = 0

        self.save_model_step = save_model_step
        self.save_figure_step = save_figure_step

        self.device = device
        self.save_dir = save_dir

        self.num_envs = num_envs
        self.envs = envs
        self.additional_bonus_state_8_4_option = additional_bonus_state_8_4_option
        self.model = model.to(self.device)
        self.target_model = target_model.to(self.device)

        self.learning_rate = learning_rate
        self.gamma = gamma
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate, eps = 1e-3)

        #self.model = torch.compile(self.model)

        self.m = m
        self.l = l
        self.n = n
        self.epsilons = epsilons
        self.start_learning_step = start_learning_step
        self.start_learning_sequence = start_learning_sequence

        self.per_eps = per_eps
        self.per_alpha = per_alpha
        self.per_beta = per_beta
        self.eta = eta
        self.replay_buffer_size = replay_buffer_size
        self.per = per

        self.max_grad_norm = max_grad_norm
        self.batch_size = batch_size
        self.is_completed = False

        self.env = None
        self.max_test_score = -1e9
        self.loss_type = loss_type

        # I just log 1000 lastest update and print it to log.
        self.losses = np.zeros((1000,)).reshape(-1)
        self.loss_index = 0
        self.len_loss = 0

    def save_figure(self, is_training = False):
        # test current model and save model/figure if model yield best total rewards.
        # create env for testing, reset test env
        if self.env is None:
            self.env = create_env(self.world, self.stage, self.action_type, self.additional_bonus_state_8_4_option, True)
        state = self.env.reset()
        done = False

        images = []
        total_reward = 0
        total_step = 0
        num_repeat_action = 0
        old_action = -1

        episode_time = datetime.now()

        # create h, c as zeros
        h = torch.zeros((1, 512), dtype=torch.float, device = self.device)
        c = torch.zeros((1, 512), dtype=torch.float, device = self.device)

        last_actions = torch.zeros((1, self.action_dim), device = self.device)
        last_rewards = torch.zeros((1,), device = self.device).reshape(-1)

        # play 1 episode, just get loop action with max probability from model until the episode end.
        while not done:
            with torch.no_grad():
                Q, h, c = self.model(torch.tensor(state, dtype = torch.float, device = self.device).unsqueeze(0), h, c,
                                     last_actions, last_rewards.unsqueeze(-1))

            action = Q.argmax(-1).item()
            next_state, reward, done, trunc, info = self.env.step(action)
            state = next_state
            img = Image.fromarray(self.env.current_state)
            images.append(img)
            total_reward += reward
            total_step += 1

            last_actions = F.one_hot(torch.as_tensor(np.array([action]), device = self.device), self.action_dim).to(self.device)
            last_rewards[0] = reward

            if action == old_action:
                num_repeat_action += 1
            else:
                num_repeat_action = 0
            old_action = action
            if num_repeat_action == 200:
                break

        #logging, if model yield better result, save figure (test_episode.mp4) and model (best_model.pth)
        if is_training:
            f_out = open(f"logging_test.txt", "a")
            f_out.write(f'episode_reward: {total_reward:.4f} episode_step: {total_step} current_step: {self.current_step} \
loss: {(self.losses.sum()/self.len_loss):.4f} len_per: {self.per.real_size} episode_time: {datetime.now() - episode_time}\n')
            f_out.close()

        if total_reward > self.max_test_score or info['flag_get']:
            imageio.mimsave('test_episode.mp4', images)
            self.max_test_score = total_reward
            if is_training:
                torch.save(self.model.state_dict(), f"best_model.pth")

        # if model can complete this game, stop training by set self.is_completed to True
        if info['flag_get']:
            self.is_completed = True

    def save_model(self):
        torch.save(self.model.state_dict(), f"model_{self.current_step}.pth")

    def load_model(self, model_path = None):
        if model_path is None:
            model_path = f"model_{self.current_step}.pth"
        self.model.load_state_dict(torch.load(model_path))

    def update_loss_statis(self, loss):
        # update loss for logging, just save 1000 latest updates.
        self.losses[self.loss_index] = loss
        self.loss_index = (self.loss_index + 1)%1000
        self.len_loss = min(self.len_loss+1, 1000)

    def select_action(self, states, h, c, target_h, target_c, last_actions, last_rewards):
        states = torch.as_tensor(np.array(states), device = self.device)

        with torch.no_grad():
            Q, h, c = self.model(states, h, c, last_actions, last_rewards.unsqueeze(-1))
            target_q, target_h, target_c = self.target_model(states, target_h, target_c, last_actions, last_rewards.unsqueeze(-1))
            actions = Q.argmax(-1)

            n_actions = Q.shape[1]
            batch_size = states.shape[0]
            random_actions = torch.randint(0, n_actions, (batch_size,), device=self.device)

            use_random = torch.rand(batch_size, device=self.device) < self.epsilons
            actions = torch.where(use_random, random_actions, actions)

        return actions, Q, target_q, h, c, target_h, target_c

    def update_target_model(self):
        with torch.no_grad():
            for target_param, online_param in zip(self.target_model.parameters(), self.model.parameters()):
                target_param.copy_(online_param)
        gc.collect()
        torch.cuda.empty_cache()

    @staticmethod
    def value_rescale(value, eps=1e-3):
        return value.sign()*((value.abs()+1).sqrt()-1) + eps*value

    @staticmethod
    def inverse_value_rescale(value, eps=1e-3):
        temp = ((1 + 4*eps*(value.abs()+1+eps)).sqrt() - 1) / (2*eps)
        return value.sign() * (temp.square() - 1)

    def learn(self):
        # get all data
        # states: (m+l+n) x b x 1 x 84 x 84
        # actions: (m+l+n) x b
        # rewards: (m+l+n) x b
        # dones: (m+l+n) x b
        # init_h: b x 512
        # init_c: b x 512
        # last_actions: b x 512 x a
        # last_rewards: b x 512
        # masks: (m+l+n) x b
        (states, actions, rewards, dones, init_h, init_c, last_actions, last_rewards, masks), weights, sample_idx = self.per.sample(self.batch_size)
        states = torch.as_tensor(states, device=self.device)
        actions = torch.as_tensor(actions, device=self.device, dtype=torch.long)
        rewards = torch.as_tensor(rewards, device=self.device, dtype=torch.float32)
        dones = torch.as_tensor(dones, device=self.device, dtype=torch.float32)
        masks = torch.as_tensor(masks, device=self.device, dtype=torch.float32)
        weights = torch.as_tensor(weights, device=self.device, dtype=torch.float32)
        last_actions = torch.as_tensor(last_actions, device=self.device)
        last_rewards = torch.as_tensor(last_rewards, device=self.device)

        Qs, target_qs = [], []

        # burn in
        init_h, init_c = torch.tensor(init_h).to(self.device), torch.tensor(init_c).to(self.device)
        h, c = init_h.clone(), init_c.clone()
        target_h, target_c = init_h.clone(), init_c.clone()

        for t in range(self.m):
            with torch.no_grad():
                q, h, c = self.model(states[t], h, c, last_actions[t], last_rewards[t].unsqueeze(-1))
                target_q, target_h, target_c = self.target_model(states[t], target_h, target_c, last_actions[t], last_rewards[t].unsqueeze(-1))
                h, c = h * masks[t].unsqueeze(1), c * masks[t].unsqueeze(1)
                target_h, target_c = target_h * masks[t].unsqueeze(1), target_c * masks[t].unsqueeze(1)
                Qs.append(q)
                target_qs.append(target_q)

        # calculate Q and target_Q
        for t in range(self.m, self.m + self.l):
            q, h, c = self.model(states[t], h, c, last_actions[t], last_rewards[t].unsqueeze(-1))
            with torch.no_grad():
                target_q, target_h, target_c = self.target_model(states[t], target_h, target_c, last_actions[t], last_rewards[t].unsqueeze(-1))
            Qs.append(q)
            target_qs.append(target_q)

        # last n_step
        h = h.detach().clone()
        c = c.detach().clone()

        with torch.no_grad():
            for t in range(self.m + self.l, self.m + self.l + self.n):
                q, h, c = self.model(states[t], h, c, last_actions[t], last_rewards[t].unsqueeze(-1))
                target_q, target_h, target_c = self.target_model(states[t], target_h, target_c, last_actions[t], last_rewards[t].unsqueeze(-1))
                Qs.append(q)
                target_qs.append(target_q)

        # calculate target
        idx = torch.arange(0, len(h), device = self.device)
        targets = []
        with torch.no_grad():
            for t in range(self.m + self.l - 1, self.m-1, -1):
                max_action = Qs[t + self.n].argmax(-1)
                target = self.inverse_value_rescale(target_qs[t + self.n][idx, max_action])

                for k in range(t + self.n - 1, t-1, -1):
                    target = target * self.gamma * (1. - dones[k]) + rewards[k]
                targets.append(self.value_rescale(target))

        # calculate loss
        loss = 0.
        targets = targets[::-1]
        max_delta = torch.zeros((len(idx),)).reshape(-1).to(self.device)
        mean_delta = 0.
        counts = 0.
        for t in range(self.m, self.m + self.l):
            if self.loss_type == 'huber':
                delta = (F.huber_loss(Qs[t][idx, actions[t]], targets[t-self.m], reduction = "none") * masks[t])
            else:
                delta = (F.mse_loss(Qs[t][idx, actions[t]], targets[t-self.m], reduction = "none") * masks[t])
            loss += (delta.reshape(-1) * weights.reshape(-1)).sum()
            counts += masks[t]
            with torch.no_grad():
                delta = (Qs[t][idx, actions[t]] - targets[t-self.m]).abs() * masks[t]
                max_delta = torch.maximum(max_delta, delta.reshape(-1))
                mean_delta = mean_delta + delta.reshape(-1)
        mean_delta = mean_delta / counts
        loss /= counts.sum()

        #update model
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
        self.optimizer.step()

        #update loss statis
        self.update_loss_statis(loss.item())

        #update priority
        priorities = self.eta * max_delta + (1-self.eta) * mean_delta
        self.per.update_priorities(sample_idx, priorities)

    def calculate_target(self, rewards, dones, values, target_values):
        targets = []
        T = len(rewards)
        for t in range(T):
            if t + self.n < T:
                max_action = values[t + self.n].argmax(-1)
                target = self.inverse_value_rescale(target_values[t + self.n][max_action])
            else:
                target = 0

            for k in range(min(T-1, t + self.n - 1), t-1, -1):
                target = target * self.gamma * (1. - dones[k].float()) + rewards[k]
            targets.append(self.value_rescale(target))

        return torch.tensor(targets).reshape(-1)

    def calculate_priorities(self, values, targets, actions, sequence_idx):
        T = len(values)
        priorities = []
        idxs = torch.arange(0, len(values), device = self.device)
        values = values[idxs, actions]
        deltas = (values.cpu() - targets).abs()
        for i, idx in enumerate(sequence_idx):
            start = idx
            end = min(T, idx + self.l)
            delta = deltas[start:end]
            max_delta = delta.max()
            mean_delta = delta.mean()
            priorities.append(self.eta * max_delta.item() + (1-self.eta) * mean_delta.item())
        return priorities

    def save_episode(self, data):
        # (T x 1 x 84 x 84) (T) (T) (T) (T a) (T a) (T 512) (T 512) (T 512 a) (T 512) (1)
        states, actions, rewards, dones, values, target_values, h, c, last_actions, last_rewards = data

        states = np.array(states)
        actions, rewards, dones = torch.tensor(actions, dtype=torch.long).to(self.device), torch.tensor(rewards), torch.tensor(dones).to(self.device)
        values, target_values = torch.stack(values, 0).to(self.device), torch.stack(target_values, 0).to(self.device)
        h, c = torch.stack(h, 0).to(self.device), torch.stack(c, 0).to(self.device)
        last_actions, last_rewards = np.array(last_actions), np.array(last_rewards)

        T = len(states)
        sequence_idx = torch.arange(0, T, self.l).reshape(-1) # (S)
        burnin_sequence_idx = torch.clamp(sequence_idx - self.m, min=0)
        end_sequence_idx = torch.clamp(sequence_idx + self.l + self.n - 1, max=T - 1)

        targets = self.calculate_target(rewards, dones, values, target_values) # (T 512)
        priorities = self.calculate_priorities(values, targets, actions, sequence_idx) # (S)

        h, c = h[burnin_sequence_idx].detach(), c[burnin_sequence_idx].detach()

        self.per.add((states, actions.cpu(), rewards.cpu(), dones.cpu(), h.cpu(), c.cpu(), last_actions, last_rewards),
                     burnin_sequence_idx, sequence_idx, end_sequence_idx, priorities)

    def train(self):
        episode_reward = [0] * self.num_envs
        episode_step = [0] * self.num_envs
        max_episode_reward = 0
        max_episode_step = 0
        episode_time = [datetime.now() for _ in range(self.num_envs)]
        total_time = datetime.now()

        last_episode_rewards = []

        #reset envs
        states = self.envs.reset() #list n, 1, 84, 84

        # create h, c as zeros
        h = torch.zeros((self.num_envs, 512), dtype=torch.float, device = self.device) # n 512
        c = torch.zeros((self.num_envs, 512), dtype=torch.float, device = self.device) # n 512

        target_h = torch.zeros((self.num_envs, 512), dtype=torch.float, device = self.device) # n 512
        target_c = torch.zeros((self.num_envs, 512), dtype=torch.float, device = self.device) # n 512

        episode_memorys = Episode_memory(self.num_envs, self.m, self.l, self.n)

        self.update_target_model()

        last_actions = torch.zeros((self.num_envs, self.action_dim), device = self.device) # n a
        last_rewards = torch.zeros((self.num_envs,), device = self.device).reshape(-1) # n

        while True:
            # finish training if agent reach total_step or total_episode base on what type of total_step_or_episode is step or episode
            self.current_step += 1

            if self.total_step_or_episode == 'step':
                if self.current_step >= self.total_step:
                    break
            else:
                if self.current_episode >= self.total_episode:
                    break

            # (n) (n x a), (n x a), (n x 512), (n x 512), (n x 512), (n x 512)
            actions, values, target_values, new_h, new_c, new_target_h, new_target_c = self.select_action(states, h, c, target_h, target_c,
                                                                                                          last_actions, last_rewards)

            # (n, 1, 84, 84) (n, 1, 84, 84) (n,) (n,)
            next_states, next_states_, rewards, dones, truncs, infos = self.envs.step(actions.cpu().numpy())

            # save to episode memorys
            episode_memorys.save(states, actions, rewards, dones, values, target_values, h, c, last_actions, last_rewards)

            last_actions = F.one_hot(actions.long().reshape(-1),
                                     self.action_dim).to(self.device).float()
            last_rewards = torch.as_tensor(np.array(rewards), device=self.device).reshape(-1).float()

            episode_reward = [x + reward for x, reward in zip(episode_reward, rewards)]
            episode_step = [x+1 for x in episode_step]

            # reset h and c to zeros for enviroments that just ending episode, just multiply h and c with (1-dones)
            h = new_h * (1 - torch.tensor(dones, device = self.device, dtype = torch.float).reshape(-1, 1))
            c = new_c * (1 - torch.tensor(dones, device = self.device, dtype = torch.float).reshape(-1, 1))

            target_h = new_target_h * (1 - torch.tensor(dones, device = self.device, dtype = torch.float).reshape(-1, 1))
            target_c = new_target_c * (1 - torch.tensor(dones, device = self.device, dtype = torch.float).reshape(-1, 1))

             # logging after each step, if 1 episode is ending, I will log this to logging.txt
            for i, done in enumerate(dones):
                if done:
                    self.current_episode += 1
                    max_episode_reward = max(max_episode_reward, episode_reward[i])
                    max_episode_step = max(max_episode_step, episode_step[i])
                    last_episode_rewards.append(episode_reward[i])
                    f_out = open(f"logging.txt", "a")
                    f_out.write(f'episode: {self.current_episode} agent: {i} epsilon: {self.epsilons[i]:.4f} rewards: {episode_reward[i]:.4f} steps: {episode_step[i]} \
complete: {infos[i]["flag_get"]==True} mean_rewards: {np.array(last_episode_rewards[-min(len(last_episode_rewards), 100):]).mean():.4f} \
max_rewards: {max_episode_reward:.4f} max_steps: {max_episode_step} current_step: {self.current_step} loss: {(self.losses.sum()/self.len_loss):.4f} \
len_per: {self.per.real_size} episode_time: {datetime.now() - episode_time[i]} total_time: {datetime.now() - total_time}\n')
                    f_out.close()
                    episode_reward[i] = 0
                    episode_step[i] = 0
                    episode_time[i] = datetime.now()

                    self.save_episode(episode_memorys.get_data(i))
                    episode_memorys.reset(i)

                    last_actions[i] = last_actions[i] * 0
                    last_rewards[i] = 0

            # training agent every learn_step
            if (self.current_step % self.learn_step == 0 and
                self.current_step * self.num_envs >= self.start_learning_step and
                self.per.real_size >= self.start_learning_sequence and
                self.per.real_size > self.batch_size):
                self.learn()

            # update target model every target_update_freq
            if self.current_step % (self.target_update_freq * self.learn_step) == 0:
                self.update_target_model()

            # eval agent every save_figure_step
            if self.current_step % self.save_figure_step == 0:
                self.save_figure(is_training=True)
                if self.is_completed:
                    return

            if self.current_step % self.save_model_step == 0:
                self.save_model()

            states = list(next_states_)

        f_out = open(f"logging.txt", "a")
        f_out.write(f' mean_rewards: {np.array(last_episode_rewards[-min(len(last_episode_rewards), 100):]).mean()} max_rewards: {max_episode_reward} \
max_steps: {max_episode_step} current_step: {self.current_step} total_time: {datetime.now() - total_time}\n')
        f_out.close()

# Create model

In [ ]:
# change from https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/ppo_rnd_envpool.py
# ALGO LOGIC: initialize agent here:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    if isinstance(layer, (nn.LSTM, nn.LSTMCell)):
        for name, param in layer.named_parameters():
            if "weight" in name:
                torch.nn.init.orthogonal_(param, std)
            elif "bias" in name:
                torch.nn.init.constant_(param, bias_const)
    else:
        torch.nn.init.orthogonal_(layer.weight, std)
        torch.nn.init.constant_(layer.bias, bias_const)
    return layer

In [ ]:
class Model(nn.Module):
    def __init__(self, input_dim, output_dim, use_layer_init):
        super(Model, self).__init__()
        self.conv1 = layer_init(nn.Conv2d(1, 32, 8, stride=4)) if use_layer_init else nn.Conv2d(1, 32, 8, stride=4)
        self.conv2 = layer_init(nn.Conv2d(32, 64, 4, stride=2)) if use_layer_init else nn.Conv2d(32, 64, 4, stride=2)
        self.conv3 = layer_init(nn.Conv2d(64, 64, 3, stride=1)) if use_layer_init else nn.Conv2d(64, 64, 3, stride=1)
        self.linear1 = layer_init(nn.Linear(3136, 512)) if use_layer_init else nn.Linear(3136, 512)
        self.lstm = layer_init(nn.LSTMCell(512+output_dim+1, 512)) if use_layer_init else nn.LSTMCell(512+output_dim+1, 512)
        self.linearV = layer_init(nn.Linear(512, 512)) if use_layer_init else nn.Linear(512, 512)
        self.value = layer_init(nn.Linear(512, 1)) if use_layer_init else nn.Linear(512, 1)
        self.linearA = layer_init(nn.Linear(512, 512)) if use_layer_init else nn.Linear(512, 512)
        self.advantage = layer_init(nn.Linear(512, output_dim)) if use_layer_init else nn.Linear(512, output_dim)

    def forward(self, x, h, c, action, reward):
        x = F.relu(self.conv1(x/255.))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.linear1(x))

        x = torch.cat([x, action, reward], -1)
        h, c = self.lstm(x, (h, c))

        a = F.relu(self.linearA(h))
        a = self.advantage(a)

        v = F.relu(self.linearV(h))
        v = self.value(v)

        q = v + a - a.mean(1, keepdim=True)
        return q, h, c

# train

In [ ]:
model = Model(config.state_dim, config.action_dim, config.use_layer_init)
target_model = Model(config.state_dim, config.action_dim, config.use_layer_init)

In [ ]:
envs = MultipleEnvironments(config.world, config.stage, config.action_type, config.num_envs, config.additional_bonus_state_8_4_option)

In [ ]:
epsilon_base = 0.4
epsilons = torch.tensor(np.array([epsilon_base ** (1 + i / (config.num_envs - 1) * 7) for i in range(config.num_envs)]),
                        device = "cuda" if torch.cuda.is_available() else "cpu")
print(epsilons)

In [ ]:
per = R2D2ReplayBuffer(config.state_dim, config.action_dim, config.replay_buffer_size, config.replay_buffer_sample_size,
                       config.l, config.m, config.n, config.per_eps, config.per_alpha, config.per_beta)

In [ ]:
agent = Agent(world = config.world, stage = config.stage, action_type = config.action_type, envs = envs, num_envs = config.num_envs,
              additional_bonus_state_8_4_option = config.additional_bonus_state_8_4_option,
              state_dim = config.state_dim, action_dim = config.action_dim, save_dir = config.save_dir,
              save_model_step = config.save_model_step, save_figure_step = config.save_figure_step, learn_step = config.learn_step,
              total_step_or_episode = config.total_step_or_episode, total_step = config.total_step, total_episode = config.total_episode,
              model = model, target_model = target_model, gamma = config.gamma, learning_rate = config.learning_rate,
              max_grad_norm = config.max_grad_norm, target_update_freq = config.target_update_freq, replay_buffer_size = config.replay_buffer_size,
              per = per, batch_size = config.batch_size, loss_type = config.loss_type, per_eps = config.per_eps, per_alpha = config.per_alpha,
              per_beta = config.per_beta, eta = config.eta, m = config.m, l = config.l, n = config.n, epsilons = epsilons,
              start_learning_step = config.start_learning_step, start_learning_sequence = config.start_learning_sequence,
              device = "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
agent.train()

#test

In [ ]:
agent.load_model("best_model.pth")
agent.save_figure()